In [1]:
using MarineHydro
using PyCall
using DimensionalData
cpt = pyimport("capytaine")

# Description of problem
h = Inf # sea depth [m]
omegas = [1.03, 1.7] # frequencies [rad/s]
betas = [0.0, pi/6] # incident wave angle [rad]
beta = betas[1]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
len = 2.5
faces_max_radius = 0.5
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)


# Create FloatingBody object
cptbody = cpt.FloatingBody(mesh=cptmesh)
cptbody.center_of_mass = (0.0, 0.0, 0.0)
cptbody.rotation_center = (1.0, 1.0, 0.0) # off set for nonzero off-diagoinal elements
foreach(dof -> cptbody.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody.add_rotation_dof(name=dof), r_DOFs)
cptbody.active_dofs = DOFs
cptbody.name = "Horizontal Cylinder"


# Setup and solve BEM problems
solver = cpt.BEMSolver()
dof_list = cptbody.active_dofs
xr = pyimport("xarray")
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => betas, "radiating_dof" => DOFs))
results = cpt.BEMSolver().fill_dataset(test_matrix, cptbody, method="direct")
 

# Get Capytaine values
A_cpt = results.added_mass
B_cpt = results.radiation_damping
F_FK_cpt = results.Froude_Krylov_force 
F_D_cpt = results.diffraction_force
F_ex_cpt = results.excitation_force

Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.DataArray 'excitation_force' (omega: 2, wave_direction: 2,
                                      influenced_dof: 2)> Size: 128B
array([[[56324.96719669-1903.32445886j, 56302.4005031 +3306.43613383j],
        [56243.58030218-1900.79195813j, 56224.02487648+2619.21507886j]],

       [[38111.15003424-6795.65856077j, 37155.39617903+7120.21619363j],
        [37576.54544031-6732.34597004j, 36744.28632526+5468.97441522j]]])
Coordinates:
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
  * wave_direction  (wave_direction) float64 16B 0.0 0.5236
  * omega           (omega) float64 16B 1.03 1.7
  * influenced_dof  (influenced_dof) object 16B 'Heave' 'Pitch'
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33

In [2]:
Symbol.(DOFs)

2-element Vector{Symbol}:
 :Heave
 :Pitch

In [3]:
F_D_cpt

PyObject <xarray.DataArray 'diffraction_force' (omega: 2, wave_direction: 2,
                                       influenced_dof: 2)> Size: 128B
array([[[ -6743.84717853-1903.32445886j,  -6766.41387212+1473.77719903j],
        [ -6805.08594248-1900.79195813j,  -6824.64136818+1029.58051703j]],

       [[-11852.53171731-6795.65856077j, -12808.28557252+2534.67162652j],
        [-12251.29998743-6732.34597004j, -13083.55910249+1455.4278707j ]]])
Coordinates:
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
  * wave_direction  (wave_direction) float64 16B 0.0 0.5236
  * omega           (omega) float64 16B 1.03 1.7
  * influenced_dof  (influenced_dof) object 16B 'Heave' 'Pitch'
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33
Attributes:
    long_name:  Diffraction force

In [4]:
# Restructure MDOF marinehydro functions
mhmesh = Mesh(cptmesh)
num_panels = mhmesh.nfaces

# manually make heave dof matrix
dof_mat_1 = zeros(num_panels,3)
dof_mat_1[:,3].=1

# manually make surge dof matrix
dof_mat_2 = zeros(num_panels,3)
dof_mat_2[:,2].=1

# make dof named tuple
dofs = (dof1 = dof_mat_1, dof2 = dof_mat_2)


(dof1 = [0.0 0.0 1.0; 0.0 0.0 1.0; … ; 0.0 0.0 1.0; 0.0 0.0 1.0], dof2 = [0.0 1.0 0.0; 0.0 1.0 0.0; … ; 0.0 1.0 0.0; 0.0 1.0 0.0])

In [5]:
Dofs = NamedTuple
Dofs

NamedTuple

In [64]:
# redo floatingbody object
using LinearAlgebra: cross, dot, norm

struct FloatingBody
    mesh::Mesh
    dofs::NamedTuple 

    function FloatingBody(mesh::Mesh, dofs::NamedTuple)
        #add assert statements

        return new(mesh, dofs)
    end
end



# function FloatingBody(mesh::Mesh, rigid_dof_list::Vector{String}, rotation_center::AbstractVector)
#     dofs = Dict()

#     for dof_name in rigid_dof_list
#         if dof_name in ["Surge", "Sway", "Heave"]
#             dofs[dof_name] = translational_dofs(mesh, dof_name)
#         elseif dof_name in ["Roll", "Pitch", "Yaw"]
#             dofs[dof_name] = rotational_dofs(mesh, dof_name, rotation_center)
#         end
#     end
#     return FloatingBody(mesh, dofs)
# end




function FloatingBody(mesh::Mesh, rigid_dof_list::Vector{String}, rotation_center::AbstractVector)
    
    # generator
    dof_pairs = (Symbol(name) => if name in ["Surge", "Sway", "Heave"]
            translational_dofs(mesh, name)
        else
            rotational_dofs(mesh, name, rotation_center)
        end for name in rigid_dof_list)
            
    # convert Pair to NamedTuple using ; and splat
    dofs = (; dof_pairs...)

    return FloatingBody(mesh, dofs)
end

function FloatingBody(mesh::Mesh, rigid_dof_list::Vector{Symbol}, rotation_center::AbstractVector)
    return FloatingBody(mesh, string.(rigid_dof_list), rotation_center)
end

function translational_dofs(mesh::Mesh, dof_name::String)
    num_panels = mesh.nfaces
    dof = zeros(num_panels, 3)
    if dof_name=="Surge"
        dof[:,1] .= 1
    elseif dof_name=="Sway"
        dof[:,2] .= 1
    elseif dof_name=="Heave"
        dof[:,3] .= 1
    end
    return dof
end

function rotational_dofs(mesh::Mesh, dof_name::String, rotation_center::AbstractVector)
    face_centers = mesh.centers
    if dof_name=="Roll"
        axis_of_rot = [1, 0, 0]
    elseif dof_name=="Pitch"
        axis_of_rot = [0, 1, 0]
    elseif dof_name=="Yaw"
        axis_of_rot = [0, 0, 1]
    end
    pos_vec = face_centers .- rotation_center'
    dof_vecs = cross.(Ref(axis_of_rot), eachrow(pos_vec))
    dof = copy(stack(dof_vecs)') # make vector of vectors into a matrix
    return dof
end

function FloatingBody(mesh::Mesh, rigid_dof_list::Vector{String})
    rotation_center = [0.0,0.0,0.0]
    for dof in rigid_dof_list
        if dof in ["Roll","Pitch","Yaw"]
            display("Setting origin as rotation center.")
        end
    end
    return FloatingBody(mesh::Mesh, rigid_dof_list::Vector{String}, rotation_center::AbstractVector)
end

FloatingBody

In [65]:
mesh = mhmesh
rotation_center = [1.0,1.0,0.0]


fb = FloatingBody(mhmesh,DOFs,rotation_center)

FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.13558866847423695, 0.13558866847423684], [0.45723

In [67]:
FloatingBody(mhmesh,Symbol.(DOFs),rotation_center)

FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.13558866847423695, 0.13558866847423684], [0.45723

In [8]:
keys(fb.dofs)

(:Heave, :Pitch)

In [9]:
fb.dofs[:Pitch]

80×3 Matrix{Float64}:
 -1.42573    0.0   1.9375
 -1.42573    0.0   1.3125
 -1.42573    0.0   0.6875
 -1.42573    0.0   0.0625
 -1.14334    0.0   1.9375
 -1.14334    0.0   1.3125
 -1.14334    0.0   0.6875
 -1.14334    0.0   0.0625
 -0.634508   0.0   1.9375
 -0.634508   0.0   1.3125
  ⋮               
 -0.0370868  0.0  -0.25
 -0.0370868  0.0  -0.25
 -0.141002   0.0  -0.25
 -0.254076   0.0  -0.25
 -0.316828   0.0  -0.25
 -0.0865359  0.0  -0.25
 -0.0865359  0.0  -0.25
 -0.14093    0.0  -0.25
 -0.14093    0.0  -0.25

In [10]:
# constants
g = 9.81 # acceleration due to gravity [m/s^2]
rho = 1020 # fluid density [kg/m^3]

1020

In [11]:
# redo radiation_bc function

# function radiation_bc(floatingbody::FloatingBody, omega)
#     """
#     Calculates the radiation boundary conditions for floating bodies at each panel.

#     # Arguments
#     - `floatingbody::FloatingBody`: The floating body
#     - `omega`: The frequency of the incident ocean wave ~~~.

#     # Returns
#     - The (Neumann) radiation boundary condition values for each panel.
# """
#     normals_mat = floatingbody.mesh.normals

#     # generator
#     bc_pairs = (dof_symbol => begin
#         dof_mat = floatingbody.dofs[dof_symbol]
#         -1im .* omega .* sum(normals_mat .* dof_mat, dims=2) # output
#     end for dof_symbol in keys(floatingbody.dofs))    
   
#     # convert Pair to NamedTuple using ; and splat
#     bc = (; bc_pairs...)
#     return bc
# end


function radiation_bc(mesh::Mesh, dof_mat::Matrix{Float64}, omega::Real)
    """
    Calculates the radiation boundary conditions for floating bodies at each panel.

    # Arguments
    - `floatingbody::FloatingBody`: The floating body
    - `omega`: The frequency of the incident ocean wave ~~~.

    # Returns
    - The (Neumann) radiation boundary condition values for each panel.
"""
    bc =  -1im .* omega .* sum(mesh.normals .* dof_mat, dims=2)
    return bc
end




radiation_bc (generic function with 1 method)

In [12]:
radiation_bc(fb.mesh, fb.dofs[:Heave], 1.5)

80×1 Matrix{ComplexF64}:
 -0.0 + 1.4623918682727357im
 -0.0 + 1.4623918682727357im
 -0.0 + 1.4623918682727357im
 -0.0 + 1.4623918682727357im
 -0.0 + 1.1727472237020446im
 -0.0 + 1.1727472237020446im
 -0.0 + 1.1727472237020446im
 -0.0 + 1.1727472237020446im
 -0.0 + 0.6508256086763374im
 -0.0 + 0.6508256086763374im
      ⋮
  0.0 - 0.0im
  0.0 - 0.0im
  0.0 - 0.0im
  0.0 - 0.0im
  0.0 - 0.0im
  0.0 - 0.0im
  0.0 - 0.0im
  0.0 - 0.0im
  0.0 - 0.0im

In [13]:
# redo integrate ressure
# function integrate_pressure(floatingbody::FloatingBody, pressure)
#     mesh = floatingbody.mesh

#     # generator
#     force_pairs = (dof_symbol => begin
#         dof_mat = floatingbody.dofs[dof_symbol]
#         normal_dof_amp_on_face = -sum(dof_mat .* mesh.normals, dims=2)
#         sum(pressure .* normal_dof_amp_on_face .* mesh.areas) # output
#     end for dof_symbol in keys(floatingbody.dofs))

#     # convert Pair to NamedTuple using ; and splat
#     forces = (; force_pairs...)
#     return forces
# end

function integrate_pressure(floatingbody::FloatingBody, pressure)
    mesh = floatingbody.mesh

    # generator
    force_pairs = (dof_symbol => begin
        dof_mat = floatingbody.dofs[dof_symbol]
        normal_dof_amp_on_face = -sum(dof_mat .* mesh.normals, dims=2)
        sum(pressure .* normal_dof_amp_on_face .* mesh.areas) # output
    end for dof_symbol in keys(floatingbody.dofs))

    # convert Pair to NamedTuple using ; and splat
    forces = (; force_pairs...)
    return forces
end






















# check capytain's version of this function. make this output a vector if they do a vector. if they use a dict, keep it.

integrate_pressure (generic function with 1 method)

In [14]:
# dummy_pressure = ones(num_panels,1)
# forces_tuple = integrate_pressure(fb, dummy_pressure)
# vec1 = collect(ComplexF64, values(forces_tuple))
# vec2 = vec1
# vec_tuple = (vec1 = vec1, vec2 = vec2)
# mat = hcat(values(vec_tuple)...)
# real(mat)
# values(forces_tuple)

In [15]:
# Try out making struct for each type of problem

# Abstract type named LinearPotentialFlowProblem. The diffraction and radiation problems will be subtypes of this type. 
abstract type LinearPotentialFlowProblem end

# Define DiffractionProblem struct as a subtype of LinearPotentialFlowProblem
struct DiffractionProblem <: LinearPotentialFlowProblem
    floatingbody::FloatingBody
    omega::Real
    beta::Real
    function DiffractionProblem(floatingbody::FloatingBody,
        omega::Real,
        beta::Real)
        return new(floatingbody, omega, beta)
    end
end

# Define RadiationProblem struct as a subtype of LinearPotentialFlowProblem
struct RadiationProblem <: LinearPotentialFlowProblem
    floatingbody::FloatingBody
    omega::Real
    radiating_dof::Symbol
    function RadiationProblem(floatingbody::FloatingBody,
        omega::Real,
        radiating_dof::Symbol)
        @assert (radiating_dof in keys(floatingbody.dofs)) "the dof Symbol must be a key of floatingbody.dof"
        return new(floatingbody, omega, radiating_dof)
    end
end








In [16]:
dof_list = keys(fb.dofs)
dof_list[1]

:Heave

In [17]:
dif_prob = DiffractionProblem(fb,omegas[1],0.1)
rad_prob = RadiationProblem(fb,omegas[1],dof_list[1])

RadiationProblem(FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.13558866847423695, 0.13558866847

In [18]:
abstract type LinearPotentialFlowResult end

struct DiffractionResult <: LinearPotentialFlowResult
    problem::DiffractionProblem
    forces::NamedTuple
    function DiffractionResult(problem::DiffractionProblem,
        forces::NamedTuple)
        return new(problem, forces)
    end
end

struct RadiationResult <: LinearPotentialFlowResult
    problem::RadiationProblem
    forces::NamedTuple
    function RadiationResult(problem::RadiationProblem,
        forces::NamedTuple)
        return new(problem, forces)
    end
end



In [19]:
function compute_bc(problem::RadiationProblem)
    return radiation_bc(problem.floatingbody.mesh,
    problem.floatingbody.dofs[problem.radiating_dof],
    problem.omega)
end

function compute_bc(problem::DiffractionProblem)
    return AiryBC(problem.floatingbody.mesh, problem.omega, problem.beta)
end


function make_result(problem::RadiationProblem, forces::NamedTuple)
    return RadiationResult(problem,forces)
end

function make_result(problem::DiffractionProblem, forces::NamedTuple)
    return DiffractionResult(problem,forces)
end




function solve_problem(problem::LinearPotentialFlowProblem)
    k = problem.omega^2 / SETTINGS.g 
    S, D = assemble_matrix_wu(problem.floatingbody.mesh, k)
    bc = compute_bc(problem)
    potential = solve(D, S, bc)
    pressure = 1im * SETTINGS.rho * problem.omega * potential
    forces = integrate_pressure(problem.floatingbody, pressure) # NamedTuple of complex forces, where each element corresponds to a dof 
    result = make_result(problem, forces)
    return result 
end





solve_problem (generic function with 1 method)

In [20]:
dif_force = solve_problem(dif_prob)
rad_force = solve_problem(rad_prob)

RadiationResult(RadiationProblem(FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.1355886684742369

In [21]:
rad_prob.radiating_dof

:Heave

In [22]:
all_probs = [dif_prob, rad_prob]

2-element Vector{LinearPotentialFlowProblem}:
 DiffractionProblem(FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.0813

In [23]:
# Equivalent to Capytaine's solve_all() function. Add parallelization  settings here.

function solve_all_problems(problems::Vector{LinearPotentialFlowProblem})
    
    results = Vector{LinearPotentialFlowResult}(undef, length(problems))
    
    for i in 1:length(problems)
        results[i] = solve_problem(problems[i])
    end
    
    return results
end



solve_all_problems (generic function with 1 method)

In [24]:
results = solve_all_problems(all_probs)

2-element Vector{LinearPotentialFlowResult}:
 DiffractionResult(DiffractionProblem(FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320

In [25]:
collect(results[1].forces)

2-element Vector{ComplexF64}:
 -6749.567704267061 - 1903.0258753119526im
 -6772.012617292802 + 1456.0454488681778im

In [26]:
DOFs

2-element Vector{String}:
 "Heave"
 "Pitch"

In [27]:
# create empty dataset 


# struct Dataset
#     wave_frequencies::AbstractVector
#     wave_directions::AbstractVector
#     radiating_dofs::Vector{String}
#     function Dataset(wave_frequencies::AbstractVector,
#         wave_directions::AbstractVector,
#         radiating_dofs::Vector{String})
#         return new(wave_frequencies, wave_directions, radiating_dofs)
#     end
# end

parameters = (wave_frequencies=omegas,
wave_directions=betas,
radiating_dofs=Symbol.(DOFs))

parameters2 = (wave_frequencies=omegas,
radiating_dofs=Symbol.(DOFs))

parameters3 = (wave_frequencies=omegas,
wave_directions=betas)



keys(parameters)
floatingbody = fb

diffraction_problems = vec([DiffractionProblem(floatingbody, omega, beta) 
            for beta in parameters[:wave_directions], 
                omega in parameters[:wave_frequencies]])

parameters

(wave_frequencies = [1.03, 1.7], wave_directions = [0.0, 0.5235987755982988], radiating_dofs = [:Heave, :Pitch])

In [28]:
dif_prob = DiffractionProblem(fb,omegas[1],0.0)

DiffractionProblem(FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320108454216, 0.08135320108454208, 0.13558866847423695, 0.135588668

In [29]:
function problems_from_data(parameters::NamedTuple, floatingbody::FloatingBody)

    # There is at least one diffraction problem to solve
    if :wave_directions in keys(parameters)
        diffraction_problems = vec([DiffractionProblem(floatingbody, omega, beta) 
            for beta in parameters[:wave_directions], 
                omega in parameters[:wave_frequencies]])
    else
        diffraction_problems = []
    end

    # There is at least one radiation problem to solve
    if :radiating_dofs in keys(parameters)
        radiation_problems = vec([RadiationProblem(floatingbody, omega, dof) 
            for dof in parameters[:radiating_dofs], 
                omega in parameters[:wave_frequencies]])
    else
        radiation_problems = []

    end

    return vcat(diffraction_problems, radiation_problems)
end


# function fill_dataset(dataset::NamedTuple, floatingbody::FloatingBody, method::String, n_jobs::Int64, n_threads::Int64)

# A_labeled = DimArray(A, (
#     Dim{:influenced_DOFs}(dof_list), 
#     Dim{:radiating_DOFs}(dof_list), 
#     Dim{:wave_frequencies}(omegas), 
#     Dim{:wave_directions}(betas)
# ))

problems_from_data (generic function with 1 method)

In [30]:
fb.dofs[:Pitch]

80×3 Matrix{Float64}:
 -1.42573    0.0   1.9375
 -1.42573    0.0   1.3125
 -1.42573    0.0   0.6875
 -1.42573    0.0   0.0625
 -1.14334    0.0   1.9375
 -1.14334    0.0   1.3125
 -1.14334    0.0   0.6875
 -1.14334    0.0   0.0625
 -0.634508   0.0   1.9375
 -0.634508   0.0   1.3125
  ⋮               
 -0.0370868  0.0  -0.25
 -0.0370868  0.0  -0.25
 -0.141002   0.0  -0.25
 -0.254076   0.0  -0.25
 -0.316828   0.0  -0.25
 -0.0865359  0.0  -0.25
 -0.0865359  0.0  -0.25
 -0.14093    0.0  -0.25
 -0.14093    0.0  -0.25

In [31]:
parameters

(wave_frequencies = [1.03, 1.7], wave_directions = [0.0, 0.5235987755982988], radiating_dofs = [:Heave, :Pitch])

In [32]:
probs = problems_from_data(parameters, fb)

problem = probs[8]

problem.floatingbody.dofs[problem.radiating_dof]

# display(probs[5])
# display(all_probs[2])


80×3 Matrix{Float64}:
 -1.42573    0.0   1.9375
 -1.42573    0.0   1.3125
 -1.42573    0.0   0.6875
 -1.42573    0.0   0.0625
 -1.14334    0.0   1.9375
 -1.14334    0.0   1.3125
 -1.14334    0.0   0.6875
 -1.14334    0.0   0.0625
 -0.634508   0.0   1.9375
 -0.634508   0.0   1.3125
  ⋮               
 -0.0370868  0.0  -0.25
 -0.0370868  0.0  -0.25
 -0.141002   0.0  -0.25
 -0.254076   0.0  -0.25
 -0.316828   0.0  -0.25
 -0.0865359  0.0  -0.25
 -0.0865359  0.0  -0.25
 -0.14093    0.0  -0.25
 -0.14093    0.0  -0.25

In [33]:
results[1].forces

(Heave = -6749.567704267061 - 1903.0258753119526im, Pitch = -6772.012617292802 + 1456.0454488681778im)

In [34]:
inf_dofs = keys(floatingbody.dofs)

findfirst(==(:Heave),inf_dofs)

1

In [58]:
results

2-element Vector{LinearPotentialFlowResult}:
 DiffractionResult(DiffractionProblem(FloatingBody(Mesh([-1.25 -1.4623918682727355 -0.33378140093447134; -1.25 -1.4623918682727357 0.0; … ; 1.25 1.4623918682727355 -0.3337814009344717; 1.25 1.4623918682727355 -2.7755575615628914e-17], [21 33 32 12; 33 42 41 32; … ; 82 79 78 81; 59 56 55 58], [-0.9375 0.3254128043381684 -1.4257266509268143; -0.3125 0.3254128043381684 -1.4257266509268143; … ; 1.25 1.2349086887636433 -0.14092992483899916; 1.25 -1.2349086887636436 -0.14092992483899902], [0.0 0.2225209339563142 -0.9749279121818238; 0.0 0.2225209339563142 -0.9749279121818238; … ; 1.0 0.0 0.0; 1.0 0.0 0.0], [0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.41722675116808966, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.4172267511680895, 0.41722675116808955, 0.41722675116808955  …  0.054235467389694765, 0.02711773369484739, 0.02711773369484736, 0.05423546738969476, 0.05423546738969476, 0.05423546738969479, 0.08135320

In [36]:
function FroudeKrylovForce(floatingbody::FloatingBody, ω, beta=0)
    """Compute the Froude-Krylov force."""

    mesh = floatingbody.mesh
    pressure =  airy_waves_pressure(mesh.centers,  ω, beta)
    forces = integrate_pressure(floatingbody, pressure) 
    return forces 
end






FroudeKrylovForce (generic function with 2 methods)

In [37]:
keys(floatingbody.dofs)

(:Heave, :Pitch)

In [38]:
FroudeKrylovForce(floatingbody, omegas[1], betas[1])

(Heave = 63068.81437521969 + 1.4709558352063256e-13im, Pitch = 63068.814375219685 + 1832.6589348041916im)

In [39]:
function assemble_data(results::Vector{LinearPotentialFlowResult})

    problems = problems_from_data(parameters, floatingbody)
    results = solve_all_problems(problems)

    omegas = parameters[:wave_frequencies]
    betas = parameters[:wave_directions]
    inf_dofs = keys(floatingbody.dofs)
    rad_dofs = keys(floatingbody.dofs)

    added_mass_data = zeros(length(inf_dofs),
        length(rad_dofs),
        length(omegas))
    radiation_damping_data = zeros(length(inf_dofs),
        length(rad_dofs),
        length(omegas))
    excitation_force_data = zeros(ComplexF64,
        length(inf_dofs),
        length(omegas),
        length(betas))
    diffraction_force_data = zeros(ComplexF64,
        length(inf_dofs),
        length(omegas),
        length(betas))
    Froude_Krylov_force_data = zeros(ComplexF64,
        length(inf_dofs),
        length(omegas),
        length(betas))

    for result in results
        if result isa DiffractionResult
            omega = result.problem.omega
            beta = result.problem.beta
            omega_index = findfirst(==(omega),omegas)
            beta_index = findfirst(==(beta),betas)
            diffraction_force_data[:,omega_index,beta_index] = collect(result.forces)
            Froude_Krylov_force_data[:,omega_index,beta_index] = collect(FroudeKrylovForce(floatingbody,omega,beta))
            excitation_force_data[:,omega_index,beta_index] = diffraction_force_data[:,omega_index,beta_index] + Froude_Krylov_force_data[:,omega_index,beta_index]

        elseif result isa RadiationResult
            rad_dof = result.problem.radiating_dof
            omega = result.problem.omega
            rad_dof_index = findfirst(==(rad_dof),rad_dofs)
            omega_index = findfirst(==(omega),omegas)
            added_mass_data[:,rad_dof_index,omega_index] = real(collect(result.forces))/omega^2
            radiation_damping_data[:,rad_dof_index,omega_index] = imag(collect(result.forces))/omega
        end
    end      



    radiation_dims = (Dim{:influenced_dofs}(collect(inf_dofs)), 
        Dim{:radiating_dofs}(collect(rad_dofs)),
        Dim{:wave_frequencies}(omegas))

    diffraction_dims = (Dim{:influenced_dofs}(collect(inf_dofs)),
        Dim{:wave_frequencies}(collect(omegas)),
        Dim{:wave_directions}(betas))


    added_mass_array = DimArray(added_mass_data, radiation_dims)
    radiation_damping_array = DimArray(radiation_damping_data, radiation_dims)
    excitation_force_array = DimArray(excitation_force_data, diffraction_dims)
    diffraction_force_array = DimArray(diffraction_force_data, diffraction_dims)
    Froude_Krylov_force_array = DimArray(Froude_Krylov_force_data, diffraction_dims)


    data = DimStack((
        added_mass = added_mass_array,
        radiation_damping = radiation_damping_array,
        excitation_force = excitation_force_array,
        diffraction_force = diffraction_force_array,
        Froude_Krylov_force = Froude_Krylov_force_array))
    return data 
end



assemble_data (generic function with 1 method)

In [40]:
data = assemble_data(results)

┌ 2×2×2×2 DimStack ┐
├──────────────────┴───────────────────────────────────────────────────── dims ┐
  ↓ influenced_dofs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  → radiating_dofs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  ↗ wave_frequencies Sampled{Float64} [1.03, 1.7] ForwardOrdered Irregular Points,
  ⬔ wave_directions Sampled{Float64} [0.0, 0.5235987755982988] ForwardOrdered Irregular Points
├────────────────────────────────────────────────────────────────────── layers ┤
  :added_mass          eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 2×2×2
  :radiation_damping   eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 2×2×2
  :excitation_force    eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 2×2×2
  :diffraction_force   eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 2×2×2
  :Froude_Krylov_force eltype: ComplexF64 dims: influenced

In [59]:
function fill_dataset(parameters::NamedTuple, floatingbody::FloatingBody)
    problems = problems_from_data(parameters, floatingbody)
    results = solve_all_problems(problems)
    data = assemble_data(results)
    return data
end

fill_dataset (generic function with 1 method)

In [60]:
data = fill_dataset(parameters, floatingbody)

┌ 2×2×2×2 DimStack ┐
├──────────────────┴───────────────────────────────────────────────────── dims ┐
  ↓ influenced_dofs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  → radiating_dofs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  ↗ wave_frequencies Sampled{Float64} [1.03, 1.7] ForwardOrdered Irregular Points,
  ⬔ wave_directions Sampled{Float64} [0.0, 0.5235987755982988] ForwardOrdered Irregular Points
├────────────────────────────────────────────────────────────────────── layers ┤
  :added_mass          eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 2×2×2
  :radiation_damping   eltype: Float64 dims: influenced_dofs, radiating_dofs, wave_frequencies size: 2×2×2
  :excitation_force    eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 2×2×2
  :diffraction_force   eltype: ComplexF64 dims: influenced_dofs, wave_frequencies, wave_directions size: 2×2×2
  :Froude_Krylov_force eltype: ComplexF64 dims: influenced

In [63]:
data[:excitation_force][:,:,2]

┌ 2×2 DimArray{ComplexF64, 2} excitation_force ┐
├──────────────────────────────────────────────┴───────────────────────── dims ┐
  ↓ influenced_dofs Categorical{Symbol} [:Heave, :Pitch] ForwardOrdered,
  → wave_frequencies Sampled{Float64} [1.03, 1.7] ForwardOrdered Irregular Points
└──────────────────────────────────────────────────────────────────────────────┘
 ↓ →             1.03               1.7
  :Heave  56240.3-1900.59im  37549.8-6724.28im
  :Pitch  56220.8+2618.17im  36717.3+5470.4im

In [62]:
F_ex_cpt

PyObject <xarray.DataArray 'excitation_force' (omega: 2, wave_direction: 2,
                                      influenced_dof: 2)> Size: 128B
array([[[56324.96719669-1903.32445886j, 56302.4005031 +3306.43613383j],
        [56243.58030218-1900.79195813j, 56224.02487648+2619.21507886j]],

       [[38111.15003424-6795.65856077j, 37155.39617903+7120.21619363j],
        [37576.54544031-6732.34597004j, 36744.28632526+5468.97441522j]]])
Coordinates:
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       <U19 76B 'Horizontal Cylinder'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
  * wave_direction  (wave_direction) float64 16B 0.0 0.5236
  * omega           (omega) float64 16B 1.03 1.7
  * influenced_dof  (influenced_dof) object 16B 'Heave' 'Pitch'
    period          (omega) float64 16B 6.1 3.696
    wavenumber      (omega) float64 16B 0.1081 0.2946
    wavelength      (omega) float64 16B 58.1 21.33

In [45]:
# redo dif force fun

# function diffraction_force(floatingbody::FloatingBody,potential,omega)
#     pressure = 1im*SETTINGS.rho* potential * omega 
#     forces = integrate_pressure(floatingbody::FloatingBody, pressure) # this is a Dict with keys associated with influenced dofs
#     return forces  
# end

# function DiffractionForce(floatingbody::FloatingBody,ω, beta=0)
    

#     mesh = floatingbody.mesh
#     green_functions = (
#         Rankine(),
#         RankineReflected(),
#         GFWu(),
#     )
#     k = ω^2 / 9.8
#     S, D = assemble_matrices(green_functions, mesh, k)
#     bc = AiryBC(mesh, ω, beta)
#     potential = solve(D, S, bc)
#     forces_tuple = diffraction_force(floatingbody, potential,ω)
#     F_D = collect(ComplexF64, values(forces_tuple)) # convert NamedTuple to Vector
#     return F_D
# end

In [46]:
# F_D_mh = DiffractionForce(fb,omegas[1], beta)

In [47]:
# F_D_cpt

In [48]:
# dof_list = collect(keys(fb.dofs))
# data_array = reshape(A_mh, size(A_mh)..., 1)
# data_labeled = DimArray(data_array, (
#         Dim{:influenced_DOFs}(dof_list), 
#         Dim{:radiating_DOFs}(dof_list),
#         Dim{:wave_frequency}([1.2])
#     ))

In [49]:
# # convert added mass and damping matrices
# function label_output(floatingbody::FloatingBody, data_array::Array{Float64, 3}, omegas)
#     dof_list = collect(keys(floatingbody.dofs))
#     num_dofs = length(dof_list)
#     num_omegas = length(omegas)

#     data_labeled = DimArray(data_array, (
#         Dim{:influenced_DOFs}(dof_list), 
#         Dim{:radiating_DOFs}(dof_list),
#         Dim{:wave_frequency}(omegas)
#     ))
#     return data_labeled
# end

# # convert single frequency added mass and damping
# function label_output(floatingbody::FloatingBody, data_mat::Matrix{Float64}, omega)
#     data_array = reshape(data_mat, size(data_mat)..., 1)
#     return label_output(floatingbody, data_array, [omega])
# end

# # convert excitation, FK, or diffraction force
# function label_output(floatingbody::FloatingBody, data_array::Array{ComplexF64, 3}, omegas, betas)
#     dof_list = collect(keys(floatingbody.dofs))
#     num_dofs = length(dof_list)
#     num_betas = length(betas)
#     num_omegas = length(omegas)

#     data_labeled = DimArray(data_array, (
#         Dim{:influenced_DOFs}(dof_list), 
#         Dim{:wave_frequency}(omegas),
#         Dim{:wave_direction}(betas)
#     ))
#     return data_labeled
# end

# # convert single frequency and beta excitation, FK, or diffraction force
# function label_output(floatingbody::FloatingBody, data_mat::Vector{ComplexF64}, omega, beta)
#     data_array = reshape(data_mat, size(data_mat)..., 1, 1)
#     return label_output(floatingbody::FloatingBody, data_array::Array{ComplexF64, 3}, [omega], [beta])
# end



In [50]:
# A_mh_labeled = label_output(fb, A_mh, omegas[1])

In [51]:
# F_D_mh_labeled = label_output(fb, F_D_mh, omegas[1], 0.0)

In [52]:
# Output

# data = rand(2, 3, 5)

# # 2. Define the dimensions with coordinates
# da = DimArray(data, (
#     X(["Heave","Surge"]),            # Longitude
#     Y(1:3),            # Latitude
#     Z([10, 50, 100, 250, 500]) # Vertical levels in meters
# ))

# da[X(At("Heave")),Y(At(1)),Z(At(10))]

In [53]:
# dof_list = ["Surge","Heave","Roll"]
# num_dofs = length(dof_list)
# betas = range(0,pi,length=3)
# num_betas = length(beta)
# omegas = range(0.2,1.5,length=10)
# num_omegas = length(omegas)


# A = rand(num_dofs,num_dofs,num_omegas,num_betas)

# A_labeled = DimArray(A, (
#     Dim{:influenced_DOFs}(dof_list), 
#     Dim{:radiating_DOFs}(dof_list), 
#     Dim{:wave_frequencies}(omegas), 
#     Dim{:wave_directions}(betas)
# ))


In [54]:
# Matrix(A_labeled[:,:,1,1])

In [55]:
# A_labeled[1,1,1,1]=0
# Matrix(A_labeled[:,:,1,1])

In [56]:
# dof_list= ["Heave", "Surge"]
# function A_mat(x)
#     A_data = [x^2 3*x; 1/x x]
#     A_labeled = DimArray(A_data, (
#         Dim{:influenced_DOFs}(dof_list), 
#         Dim{:radiating_DOFs}(dof_list)
#     ))
#     return Matrix(A_labeled)
# end

# A_mat(1.5)

# Zygote.jacobian(Omega + 0im)